# Notebook 02: Thử nghiệm và Đánh giá Sai số Mô hình (Experiments)
**Môn học:** Chuỗi khối và công nghệ sổ cái phân tán  
**Tác giả phụ trách:** Nam Khánh  
**Mục tiêu:** Chạy thử nghiệm Backtesting, so sánh giá ước tính của các mô hình và Ensemble với giá thực tế và Chainlink Oracle, tính toán MAE, RMSE, MAPE.

Kết quả (bảng, hình) được ghi vào `reports/tables/` và `reports/figures/` để dùng cho Chương 5.

In [1]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.models import NVTValuationModel, MVPQValuationModel, TWAPValuationModel, ComparableValuationModel, EnsembleValuationModel
from src.evaluation import ValuationMetrics, BacktestEngine, ChainlinkComparator, load_market_data

TABLES, FIGS = ROOT / 'reports/tables', ROOT / 'reports/figures'
TABLES.mkdir(parents=True, exist_ok=True); FIGS.mkdir(parents=True, exist_ok=True)
print('San sang chay cac thu nghiem danh gia mo hinh.')

San sang chay cac thu nghiem danh gia mo hinh.


## 5.1 Thiết lập thử nghiệm
Tập token: UNI, AAVE, LINK, MKR · 365 phiên · Train 70% / Test 30% · Baseline SMA-30.

In [2]:
TOKENS = ['UNI', 'AAVE', 'LINK', 'MKR']
TRAIN_RATIO = 0.7
market = load_market_data()
print('Nguon du lieu:', market.attrs.get('source'))
data = market[market.token.isin(TOKENS)].copy()
data.groupby('token').agg(sector=('sector','first'), n_days=('date','count'), start=('date','min'), end=('date','max'))

Nguon du lieu: synthetic


,sector,n_days,start,end
token,,,,
AAVE,Lending,365,2025-11-28,2026-11-27
LINK,Infrastructure,365,2025-11-28,2026-11-27
MKR,Lending,365,2025-11-28,2026-11-27
UNI,DEX,365,2025-11-28,2026-11-27


In [3]:
def make_models():
    return [NVTValuationModel(), MVPQValuationModel(), TWAPValuationModel(), ComparableValuationModel()]

def make_all():
    return make_models() + [EnsembleValuationModel(make_models())]

engine = BacktestEngine(train_ratio=TRAIN_RATIO, extended_metrics=True)

## 5.3 Đánh giá sai số – Hold-out 70/30

In [4]:
rows = []
for tok in TOKENS:
    g = data[data.token == tok].reset_index(drop=True)
    tbl = engine.compare_models(make_all(), g).reset_index()
    tbl.insert(0, 'token', tok)
    rows.append(tbl)
holdout = pd.concat(rows, ignore_index=True)
holdout.to_csv(TABLES / 'backtest_holdout.csv', index=False)
holdout

,token,model,MAE,RMSE,MAPE,sMAPE,Bias,R2,DirAcc
0,UNI,Ensemble_Combined,0.3333,0.4022,3.81,3.82,-0.0544,0.8050,60.55
1,UNI,Uniswap_TWAP,0.3451,0.4211,3.95,3.96,-0.0434,0.7863,62.39
2,UNI,Comparable_MCTVL,0.4860,0.5985,5.38,5.55,-0.3308,0.5682,62.39
3,UNI,Baseline_SMA30,0.7082,0.8291,8.04,8.26,-0.3158,0.1713,49.54
4,UNI,NVT_Metcalfe,1.1077,1.4098,12.67,12.48,0.1965,-1.3959,61.47
5,UNI,MV_PQ,2.0574,2.7090,23.30,20.98,0.9027,-7.8461,54.13
6,AAVE,Ensemble_Combined,4.4882,5.3938,3.72,3.67,1.3381,0.8397,48.62
7,AAVE,Uniswap_TWAP,4.6579,5.8148,3.85,3.82,1.1448,0.8137,63.30
8,AAVE,Comparable_MCTVL,6.8964,8.4363,5.77,5.73,0.5844,0.6079,72.48
9,AAVE,Baseline_SMA30,8.5194,10.5651,7.29,6.96,5.5013,0.3851,55.96


In [5]:
summary = holdout.groupby('model')[['MAE','RMSE','MAPE','sMAPE','R2','DirAcc']].mean().sort_values('MAPE')
summary.round(3).to_csv(TABLES / 'backtest_summary_by_model.csv')
summary.round(3)

,MAE,RMSE,MAPE,sMAPE,R2,DirAcc
model,,,,,,
Ensemble_Combined,16.506,20.033,3.825,3.800,0.841,54.585
Uniswap_TWAP,16.282,21.549,3.915,3.905,0.818,61.928
Comparable_MCTVL,21.725,29.387,5.198,5.182,0.683,65.600
Baseline_SMA30,37.029,49.799,7.868,7.738,0.349,50.228
NVT_Metcalfe,52.001,68.038,12.632,12.400,-0.980,50.230
MV_PQ,99.323,130.208,23.200,21.208,-5.884,49.770


## Walk-forward (expanding window, fit lại mỗi 7 ngày)

In [6]:
wf_rows = []
for tok in TOKENS:
    g = data[data.token == tok].reset_index(drop=True)
    tbl = engine.compare_models(make_all(), g, walk_forward=True, step=7).reset_index()
    tbl.insert(0, 'token', tok)
    wf_rows.append(tbl)
walk_forward = pd.concat(wf_rows, ignore_index=True)
walk_forward.to_csv(TABLES / 'backtest_walk_forward.csv', index=False)
walk_forward.groupby('model')[['MAE','MAPE','R2']].mean().sort_values('MAPE').round(3)

,MAE,MAPE,R2
model,,,
Uniswap_TWAP,11.785,2.635,0.911
Ensemble_Combined,13.921,2.870,0.911
Comparable_MCTVL,20.332,5.095,0.688
Baseline_SMA30,37.029,7.868,0.349
NVT_Metcalfe,52.580,12.640,-0.991
MV_PQ,99.239,22.910,-5.339


## 5.2 So sánh với giá thị trường và Chainlink Oracle
Chuỗi Chainlink lịch sử được mô phỏng theo cơ chế deviation threshold 1% + heartbeat.

In [7]:
cmp_rows = []
fig, axes = plt.subplots(len(TOKENS), 1, figsize=(11, 3.2 * len(TOKENS)), sharex=False)
for ax, tok in zip(axes, TOKENS):
    g = data[data.token == tok].reset_index(drop=True)
    train, test = engine.split(g)
    ens = EnsembleValuationModel(make_models()).fit(train)
    fair = pd.Series(ens.predict(test).values, index=test['date'])
    mkt = test.set_index('date')['price_usd']
    oracle = ChainlinkComparator.simulate_oracle(mkt)
    s = ChainlinkComparator.summarize(fair, mkt, oracle)
    cmp_rows.append({'token': tok, **{k: v for k, v in s.items()}, **{f'w_{k}': round(v, 3) for k, v in ens.weights.items()}})
    ax.plot(mkt.index, mkt, label='Thị trường (Spot)', lw=1.2)
    ax.plot(oracle.index, oracle, label='Chainlink (mô phỏng)', lw=1, ls='--')
    ax.plot(fair.index, fair, label='Ensemble', lw=1.6)
    ax.set_title(f'{tok} – tập kiểm tra'); ax.set_ylabel('USD'); ax.grid(alpha=.3)
axes[0].legend(loc='upper left')
fig.tight_layout(); fig.savefig(FIGS / 'ensemble_vs_market_chainlink.png', dpi=150)
chainlink_cmp = pd.DataFrame(cmp_rows).set_index('token')
chainlink_cmp.to_csv(TABLES / 'chainlink_comparison.csv')
chainlink_cmp

,n_obs,mean_model_vs_market_%,mean_abs_model_vs_market_%,mean_oracle_vs_market_%,mean_abs_model_vs_oracle_%,corr_model_market,corr_model_oracle,corr_oracle_market,MAPE_model_vs_market,MAPE_model_vs_oracle,w_NVT_Metcalfe,w_MV_PQ,w_Uniswap_TWAP,w_Comparable_MCTVL
token,,,,,,,,,,,,,,
UNI,110,-0.428,3.810,-0.003,3.785,0.8994,0.9008,0.9996,3.810,3.785,0.013,0.008,0.903,0.076
AAVE,110,1.175,3.715,-0.013,3.750,0.9262,0.9247,0.9998,3.715,3.750,0.096,0.000,0.613,0.290
LINK,110,-0.096,3.848,0.001,3.810,0.8850,0.8863,0.9998,3.848,3.810,0.039,0.030,0.761,0.171
MKR,110,1.894,3.918,-0.030,3.933,0.9726,0.9725,0.9999,3.918,3.933,0.140,0.005,0.716,0.139


## Biểu đồ so sánh MAPE theo mô hình

In [8]:
pivot = holdout.pivot(index='token', columns='model', values='MAPE')
ax = pivot.plot(kind='bar', figsize=(11, 4.5), rot=0)
ax.set_ylabel('MAPE (%)'); ax.set_title('MAPE out-of-sample theo token và mô hình'); ax.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.savefig(FIGS / 'mape_by_model.png', dpi=150)
pivot.round(2)

model,Baseline_SMA30,Comparable_MCTVL,Ensemble_Combined,MV_PQ,NVT_Metcalfe,Uniswap_TWAP
token,,,,,,
AAVE,7.29,5.77,3.72,23.85,13.80,3.85
LINK,7.17,4.59,3.85,22.34,11.89,4.09
MKR,8.97,5.05,3.92,23.31,12.17,3.77
UNI,8.04,5.38,3.81,23.30,12.67,3.95


## Định giá tương đối theo ngành (MC/TVL)

In [9]:
latest = market[market.date == market.date.max()].reset_index(drop=True)
comp = ComparableValuationModel().fit(latest)
rel = comp.relative_valuation(latest).sort_values('upside_%', ascending=False)
rel.round(4).to_csv(TABLES / 'comparable_relative_valuation.csv', index=False)
print('Benchmark MC/TVL theo nganh:', {k: round(v, 3) for k, v in comp.sector_benchmarks.items()})
rel.round(3)

Benchmark MC/TVL theo nganh: {'DEX': 0.55, 'Derivatives': 1.423, 'Infrastructure': 3.117, 'Lending': 0.176, 'Liquid Staking': 0.064}


,token,sector,mc_tvl,benchmark_mc_tvl,fair_price,price_usd,upside_%
11,ARB,Infrastructure,1.103,3.117,4.617,1.634,182.605
3,BAL,DEX,0.257,0.550,10.194,4.759,114.219
1,CRV,DEX,0.302,0.550,1.162,0.638,82.190
4,AAVE,Lending,0.136,0.176,122.929,94.906,29.527
9,LDO,Liquid Staking,0.064,0.064,2.019,2.019,0.000
5,COMP,Lending,0.176,0.176,55.604,55.604,0.000
8,GRT,Infrastructure,3.117,3.117,0.297,0.297,0.000
10,SNX,Derivatives,1.423,1.423,3.009,3.009,-0.000
6,MKR,Lending,0.217,0.176,1036.039,1280.397,-19.085
2,SUSHI,DEX,0.799,0.550,1.227,1.781,-31.088


## Nhận xét
- Ensemble tối ưu trọng số (SLSQP, cực tiểu MAE trên tập train) cho sai số thấp nhất ở hầu hết token và vượt trội baseline SMA-30.
- TWAP và Comparable bám sát giá thị trường nhất; NVT và MV=PQ phản ánh "giá trị nền tảng" nên lệch nhiều hơn khi giá bị chi phối bởi đầu cơ.
- Walk-forward cho sai số thấp hơn hold-out vì tham số được cập nhật định kỳ, phù hợp với chuỗi giá không dừng.
- **Lưu ý:** TWAP dùng giá cùng ngày trong cửa sổ trượt nên có lợi thế thông tin; khi dữ liệu thật sẵn sàng (`data/processed/market_data.csv`) cần chạy lại notebook.